# Step 8 & 9: Real-Time Stream Simulation & Prediction Storage
## Historical Data Replay / Simulated Real-Time Stream

> **IMPORTANT DISCLAIMER**: This is a **historical data replay / simulated real-time stream** using the test period (January 26–31, 2025). It is NOT live streaming NYC TLC data.

### Simulation Lifecycle:
For each 15-minute interval $t$ in the test period:
1. Clock advances to interval $t$.
2. State buffer supplies feature lags strictly available prior to $t$ ($t-1$ and older).
3. Generate 15-minute ahead predictions for all 262 pickup zones for interval $t$.
4. Advance clock; actual demand $y_t$ arrives.
5. Compute error $e_t = \hat{y}_t - y_t$ and absolute error $|e_t|$.
6. Persist structured record to the prediction audit store:
   `[prediction_timestamp, zone, prediction_horizon, predicted_demand, actual_demand, error, absolute_error, model_version]`
7. Update state buffer with $y_t$.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_processed_demand, split_chronological
from src.features import build_feature_pipeline, get_feature_columns
from src.config import MODELS_DIR, STREAM_PREDICTIONS_PARQUET

print("Loading Champion Model (model_v1)...")
model = joblib.load(MODELS_DIR / "model_v1.joblib")
feature_cols = get_feature_columns()

grid_df = load_processed_demand()
feat_df = build_feature_pipeline(grid_df, drop_burn_in=True)
_, _, test_df = split_chronological(feat_df)

print(f"Test stream replay spans: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"Total events in stream replay: {len(test_df):,} predictions")


### 1. Simulated Real-Time Replay Loop
We iterate interval-by-interval across timestamps in chronological order, simulating real-time inference and delayed ground-truth arrival.


In [ ]:
unique_timestamps = sorted(test_df['timestamp'].unique())
print(f"Number of 15-minute streaming time intervals: {len(unique_timestamps)}")

records = []
# Replay stream interval by interval
for i, ts in enumerate(unique_timestamps):
    # Interval batch arrives: only features up to ts are accessible
    interval_data = test_df[test_df['timestamp'] == ts]
    
    # 1. Feature vectors for this interval
    X_curr = interval_data[feature_cols]
    
    # 2. Predict next 15 minutes
    preds = np.maximum(0.0, model.predict(X_curr))
    
    # 3. Ground truth demand arrives
    actuals = interval_data['demand'].to_numpy()
    zones = interval_data['PULocationID'].to_numpy()
    
    # 4. Compute error metrics
    errors = preds - actuals
    abs_errors = np.abs(errors)
    
    for z, p, a, e, ae in zip(zones, preds, actuals, errors, abs_errors):
        records.append({
            'prediction_timestamp': ts,
            'zone': int(z),
            'prediction_horizon': '15min',
            'predicted_demand': round(float(p), 2),
            'actual_demand': int(a),
            'error': round(float(e), 2),
            'absolute_error': round(float(ae), 2),
            'model_version': 'model_v1'
        })
        
    if (i + 1) % 100 == 0 or (i + 1) == len(unique_timestamps):
        print(f"Replayed {i + 1}/{len(unique_timestamps)} intervals ({len(records):,} predictions logged)...")


### 2. Verify Structured Prediction Audit Store


In [ ]:
pred_df = pd.DataFrame(records)
print(f"Stream simulation completed. Total logged records: {len(pred_df):,}")
print("\nSchema and Sample Records:")
print(pred_df.info())
pred_df.head(10)


### 3. Persist Prediction Audit Store to Parquet


In [ ]:
STREAM_PREDICTIONS_PARQUET.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_parquet(STREAM_PREDICTIONS_PARQUET, index=False, engine='pyarrow', compression='snappy')
print(f"Saved prediction audit table to {STREAM_PREDICTIONS_PARQUET} (Size: {STREAM_PREDICTIONS_PARQUET.stat().st_size / 1e6:.2f} MB)")
